In [3]:
import os
import cv2
import numpy as np
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [4]:
# -----------------------------
# 1. Paths & model setup
# -----------------------------
IMAGE_DIR = "images"
OUTPUT_DIR = "output"

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cpu"  # change to "cuda" if you have a GPU

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# 2. Initialize SAM
# -----------------------------
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam)

In [5]:
# -----------------------------
# 3. Process images one by one
# -----------------------------
for image_file in os.listdir(IMAGE_DIR):
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(IMAGE_DIR, image_file)
    image_name = os.path.splitext(image_file)[0]

    print(f"\nProcessing: {image_file}")

    # Create output folder for this image
    image_output_dir = os.path.join(OUTPUT_DIR, image_name)
    os.makedirs(image_output_dir, exist_ok=True)

    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Generate masks
    masks = mask_generator.generate(image)
    print(f"Detected {len(masks)} segments")

    # -----------------------------
    # 4. Crop and save segments
    # -----------------------------
    for i, mask_data in enumerate(masks):
        mask = mask_data["segmentation"]

        # Black background
        segmented = np.zeros_like(image)
        segmented[mask] = image[mask]

        x, y, w, h = mask_data["bbox"]
        crop = segmented[int(y):int(y+h), int(x):int(x+w)]

        crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)

        output_path = os.path.join(
            image_output_dir, f"segment_{i}.png"
        )
        cv2.imwrite(output_path, crop_bgr)

    print(f"Saved to: {image_output_dir}")

print("All images processed.")


Processing: bhaat_001.jpg
Detected 121 segments
Saved to: output\bhaat_001

Processing: bhaat_002.jpg
Detected 126 segments
Saved to: output\bhaat_002

Processing: bhaat_003.jpg
Detected 33 segments
Saved to: output\bhaat_003

Processing: bhaat_004.jpg
Detected 30 segments
Saved to: output\bhaat_004

Processing: bhaat_005.jpg
Detected 37 segments
Saved to: output\bhaat_005

Processing: boiled_egg_001.jpg
Detected 42 segments
Saved to: output\boiled_egg_001

Processing: boiled_egg_002.jpg
Detected 25 segments
Saved to: output\boiled_egg_002

Processing: boiled_egg_003.jpg
Detected 16 segments
Saved to: output\boiled_egg_003

Processing: boiled_egg_004.jpg
Detected 30 segments
Saved to: output\boiled_egg_004

Processing: boiled_egg_005.jpg
Detected 4 segments
Saved to: output\boiled_egg_005

Processing: download.jpeg
Detected 108 segments
Saved to: output\download

Processing: finger_001.jpg
Detected 3 segments
Saved to: output\finger_001

Processing: finger_002.jpg
Detected 14 segments